In [41]:
import polars as pl
import psycopg2

In [42]:
df = pl.read_parquet("./data/active_genes.parquet")
# Connect to petadex
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [43]:
cur.execute(
"""
SELECT "90pid_enzyme_id", COUNT(DISTINCT logan_orfs.orf_id) AS num_orfs
FROM
    (SELECT * 
     FROM logan_catalytic_orfs
     WHERE library_id IN %s) AS logan_orfs
JOIN
    (SELECT orf_id, "90pid_enzyme_id"
     FROM petadex_clustering) AS pc
ON logan_orfs.orf_id = pc.orf_id
GROUP BY "90pid_enzyme_id";
""", (tuple(df["gene"].to_list()),))

In [44]:
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df = pl.DataFrame(rows, schema=columns, orient="row")
df.write_parquet("./data/cluster_scores.parquet")

In [45]:
df.sort('num_orfs', descending=True)

90pid_enzyme_id,num_orfs
i64,i64
1727274,31
771492,28
709802,21
1680404,19
289630,17
…,…
18172735,1
18172816,1
18172840,1
